# Displacement Then Qubit Spectroscopy


This notebook rewrites `examples/displacement_qubit_spectroscopy.py` as a guided number-splitting tutorial. We first prepare a calibrated cavity displacement with target $\alpha = 2$, then apply a long weak Gaussian qubit probe and read out the final excited-state population.

The physical signature is photon-number-dependent qubit spectroscopy: with dispersive `chi < 0`, each extra cavity photon shifts the qubit transition to lower transition detuning in the matched rotating frame. When the probe is selective enough, the resolved peak heights directly track the displaced cavity's Fock-state weights.


## Imports


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from tutorials.workflow_tutorial_support import configure_notebook_style

configure_notebook_style()

from functools import partial

from cqed_sim import (
    DisplacementGate,
    DispersiveTransmonCavityModel,
    FrameSpec,
    Pulse,
    SequenceCompiler,
    SimulationConfig,
    build_displacement_pulse,
    carrier_for_transition_frequency,
    reduced_cavity_state,
    simulate_sequence,
)
from cqed_sim.pulses.envelopes import gaussian_envelope
from tutorials.tutorial_support import (
    GHz,
    MHz,
    angular_to_mhz,
    gaussian_selective_spectrum_response,
    ns,
    us,
)


## Physics / model definition


In [ ]:
model = DispersiveTransmonCavityModel(
    omega_c=GHz(5.0),
    omega_q=GHz(6.0),
    alpha=MHz(-220.0),
    chi=MHz(-2.84),
    kerr=MHz(-0.002),
    n_cav=18,
    n_tr=2,
)
frame = FrameSpec(omega_c_frame=model.omega_c, omega_q_frame=model.omega_q)

target_alpha = 2.0 + 0.0j
probe_duration_s = 2.5 * us
probe_gap_s = 40.0 * ns
probe_amp_rad_s = MHz(0.04)
probe_sigma_fraction = 0.18
transition_scan_mhz = np.linspace(-16.0, 2.0, 181)
fock_levels = np.arange(8, dtype=int)
predicted_lines_mhz = np.asarray(
    [angular_to_mhz(model.manifold_transition_frequency(int(n), frame=frame)) for n in fock_levels],
    dtype=float,
)


## Pulse / sequence construction


In [ ]:
displacement_pulses, displacement_drive_ops, displacement_meta = build_displacement_pulse(
    DisplacementGate(index=0, name="displace_alpha_2", re=float(np.real(target_alpha)), im=float(np.imag(target_alpha))),
    {"duration_displacement_s": 120.0 * ns},
)
displacement_duration_s = float(displacement_meta["duration_s"])
dt_s = 2.0 * ns
probe_envelope = partial(gaussian_envelope, sigma=probe_sigma_fraction)

def final_excited_population_for_detuning(detuning_mhz: float) -> float:
    probe = Pulse(
        channel="qubit",
        t0=displacement_duration_s + probe_gap_s,
        duration=probe_duration_s,
        envelope=probe_envelope,
        amp=probe_amp_rad_s,
        carrier=carrier_for_transition_frequency(MHz(detuning_mhz)),
        label=f"probe_{detuning_mhz:+.2f}_MHz",
    )
    compiled = SequenceCompiler(dt=dt_s).compile(
        displacement_pulses + [probe],
        t_end=displacement_duration_s + probe_gap_s + probe_duration_s + dt_s,
    )
    result = simulate_sequence(
        model,
        compiled,
        model.basis_state(0, 0),
        {**displacement_drive_ops, "qubit": "qubit"},
        config=SimulationConfig(frame=frame, max_step=dt_s),
    )
    return float(np.real(result.expectations["P_e"][-1]))

print("Displacement metadata:", displacement_meta)


## Simulation


In [ ]:
displacement_compiled = SequenceCompiler(dt=dt_s).compile(
    displacement_pulses,
    t_end=displacement_duration_s + dt_s,
)

displacement_only = simulate_sequence(
    model,
    displacement_compiled,
    model.basis_state(0, 0),
    displacement_drive_ops,
    config=SimulationConfig(frame=frame, max_step=dt_s),
)

rho_storage = reduced_cavity_state(displacement_only.final_state)

photon_weights = np.clip(np.real(np.diag(rho_storage.full())), 0.0, 1.0)
photon_weights = photon_weights / max(np.sum(photon_weights), 1.0e-12)
nbar = float(np.real((rho_storage * qt.num(model.n_cav)).tr()))
cavity_mean = complex((rho_storage * qt.destroy(model.n_cav)).tr())

spectroscopy_response = np.asarray(
    [final_excited_population_for_detuning(point_mhz) for point_mhz in transition_scan_mhz],
    dtype=float,
)

def peak_height_near(line_mhz: float, half_window_points: int = 4) -> float:
    center_index = int(np.argmin(np.abs(transition_scan_mhz - line_mhz)))
    lo = max(0, center_index - int(half_window_points))
    hi = min(transition_scan_mhz.size, center_index + int(half_window_points) + 1)
    return float(np.max(spectroscopy_response[lo:hi]))

selected_weights = photon_weights[: fock_levels.size]
peak_heights = np.asarray([peak_height_near(line_mhz) for line_mhz in predicted_lines_mhz], dtype=float)
normalized_peak_weights = peak_heights / max(float(np.sum(peak_heights)), 1.0e-15)
normalized_photon_weights = selected_weights / max(float(np.sum(selected_weights)), 1.0e-15)

sigma_time_s = probe_duration_s * probe_sigma_fraction
theory_unscaled = gaussian_selective_spectrum_response(
    2.0 * np.pi * 1.0e6 * transition_scan_mhz,
    2.0 * np.pi * 1.0e6 * predicted_lines_mhz,
    selected_weights,
    sigma_time_s,
)
theory_scale = float(
    np.dot(theory_unscaled, spectroscopy_response)
    / max(np.dot(theory_unscaled, theory_unscaled), 1.0e-18)
)
theory_response = gaussian_selective_spectrum_response(
    2.0 * np.pi * 1.0e6 * transition_scan_mhz,
    2.0 * np.pi * 1.0e6 * predicted_lines_mhz,
    selected_weights,
    sigma_time_s,
    scale=theory_scale,
)

print("Post-displacement <n>:", nbar)
print("Post-displacement <a>:", cavity_mean)
print("Predicted n-resolved qubit lines [MHz]:", predicted_lines_mhz)


## Analysis / visualization


In [ ]:
fig, (ax_spec, ax_weights) = plt.subplots(1, 2, figsize=(11.0, 4.0))

ax_spec.plot(transition_scan_mhz, spectroscopy_response, color="#4C78A8", lw=1.5, label="pulse-level simulation")
ax_spec.plot(transition_scan_mhz, theory_response, "--", color="black", lw=1.1, label="weak-drive Gaussian theory")
for line_mhz, weight in zip(predicted_lines_mhz, selected_weights, strict=True):
    ax_spec.axvline(
        line_mhz,
        color="#E45756",
        alpha=min(0.95, 0.25 + 2.0 * float(weight)),
        lw=1.1,
    )
ax_spec.set_xlabel("Qubit transition detuning relative to frame (MHz)")
ax_spec.set_ylabel("Final excited-state probability $P_e$")
ax_spec.set_title("Selective Gaussian number splitting after calibrated displacement")
ax_spec.legend(loc="upper left")

width = 0.38
ax_weights.bar(fock_levels - width / 2.0, normalized_photon_weights, width=width, color="#72B7B2", label="displaced-state Fock weight")
ax_weights.bar(fock_levels + width / 2.0, normalized_peak_weights, width=width, color="#F58518", label="normalized peak height")
ax_weights.set_xlabel("Storage Fock level $n$")
ax_weights.set_ylabel("Normalized weight")
ax_weights.set_title(rf"Resolved peaks recover cavity populations, $\langle n \rangle = {nbar:.2f}$")
ax_weights.legend(loc="upper right")

plt.tight_layout()
plt.show()


## Interpretation


The cavity displacement prepares a coherent-state-like Fock distribution, so the spectroscopy curve is not a single broad feature. Each photon number manifold contributes its own line, separated by `chi`, and the long weak Gaussian probe narrows those lines enough to resolve them cleanly.

Because this repository defines `chi` as the per-photon qubit-transition pull, negative `chi` moves the `n`-resolved qubit lines to lower transition detuning. In the weak-drive limit the line envelope is accurately modeled by a weighted Gaussian sum, and the resolved peak heights recover the displaced-state photon weights.


## Variations / exercises


- Replace the Gaussian envelope with a shorter square probe and compare how the peaks merge together.
- Increase the displacement amplitude to make higher-`n` manifolds visible.
- Compare the extracted peak heights with the ideal coherent-state Poisson weights for the same target `alpha`.
